In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

INPUT_PATH = DATA_DIR / "ai_sentiment_rich_corpus.csv"
OUTPUT_PATH = DATA_DIR / "rag_chunks.parquet"

INPUT_PATH, OUTPUT_PATH


(PosixPath('data/ai_sentiment_rich_corpus.csv'),
 PosixPath('data/rag_chunks.parquet'))

In [2]:
df = pd.read_csv(INPUT_PATH)
print(df.shape)
df.head()


(460, 8)


,article_id,source,title,description,content,clean_content,sentiment,combined_text
0,1,Al Jazeera,What is the political agenda of artificial int...,"May 17, 2023 ... We do not believe that it cou...",Could AI single-handedly decide the course of ...,Could AI single-handedly decide the course of ...,1,What is the political agenda of artificial int...
1,2,Al Jazeera,Facebook reveals AI use to block 'terrorist co...,"Jun 16, 2017 ... Facebook says it has stepped ...",US company says technology used to block child...,US company says technology used to block child...,1,Facebook reveals AI use to block 'terrorist co...
2,3,Al Jazeera,Could artificial intelligence lead to world pe...,"May 30, 2017 ... Machines and artificial intel...",Can one man with terminal cancer complete his ...,Can one man with terminal cancer complete his ...,4,Could artificial intelligence lead to world pe...
3,4,Al Jazeera,Artificial intelligence without borders | US-M...,"Jun 9, 2023 ... The US-Mexico border is alread...",The US border-industrial complex has joined th...,The US border-industrial complex has joined th...,4,Artificial intelligence without borders | US-M...
4,6,Al Jazeera,"Paris AI summit: Why won't US, UK sign global ...","Feb 12, 2025 ... Why were the US and UK oppose...",While some countries want to ensure inclusion ...,While some countries want to ensure inclusion ...,4,"Paris AI summit: Why won't US, UK sign global ..."


In [9]:
def chunk_text_fast(text, chunk_size=220, overlap=40):
    """
    Fast word-based chunking without char_start/char_end.
    Returns a list of dicts with 'chunk_text'.
    """
    words = text.split()
    n = len(words)
    chunks = []

    if n == 0:
        return chunks

    start = 0
    while start < n:
        end = min(start + chunk_size, n)
        chunk_words = words[start:end]

        chunks.append({
            "chunk_text": " ".join(chunk_words)
        })

        # move window with overlap
        if end == n:
            break
        start = max(0, end - overlap)

    return chunks


In [10]:
all_chunks = []

CHUNK_SIZE = 220
OVERLAP = 40

for idx, row in df.iterrows():
    article_id = row["article_id"]
    source = row["source"]
    title = row["title"]
    text = str(row["clean_content"]).strip()

    # skip very short content
    if len(text.split()) < 50:
        continue

    chunks = chunk_text_fast(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP)

    for i, ch in enumerate(chunks):
        all_chunks.append({
            "chunk_id": f"{article_id}_{i}",
            "article_id": article_id,
            "source": source,
            "title": title,
            "chunk_index": i,
            "chunk_text": ch["chunk_text"],
        })

len(all_chunks)


2173

In [11]:
chunks_df = pd.DataFrame(all_chunks)
chunks_df.to_parquet(OUTPUT_PATH, index=False)

print("Saved chunked RAG corpus to:", OUTPUT_PATH)
print("Number of chunks:", len(chunks_df))
chunks_df.head()


Saved chunked RAG corpus to: data/rag_chunks.parquet
Number of chunks: 2173


,chunk_id,article_id,source,title,chunk_index,chunk_text
0,1_0,1,Al Jazeera,What is the political agenda of artificial int...,0,Could AI single-handedly decide the course of ...
1,1_1,1,Al Jazeera,What is the political agenda of artificial int...,1,"of hyperrealistic, AI-generated content, such ..."
2,1_2,1,Al Jazeera,What is the political agenda of artificial int...,2,all era-defining technology – comes with consi...
3,1_3,1,Al Jazeera,What is the political agenda of artificial int...,3,"like 2001: A Space Odyssey, Blade Runner and T..."
4,1_4,1,Al Jazeera,What is the political agenda of artificial int...,4,a clear distinction between symbolic machine ‘...
